# Graph RAG with Neo4j + Ollama

Knowledge-graph RAG pipeline using **Ollama** instead of OpenAI.

**Prerequisites:**
1. [Ollama](https://ollama.com) running locally (`ollama serve`)
2. Pull models: `ollama pull llama3.2` and `ollama pull nomic-embed-text`
3. Set Neo4j env vars (see configuration cell below)

In [2]:
%pip install --upgrade --quiet langchain langchain-community langchain-ollama langchain-neo4j langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs requests python-dotenv pip-system-certs certifi

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from typing import List, Optional, Tuple

# Fixes Neo4j Aura SSL certificate errors on Windows
try:
    import pip_system_certs  # noqa: F401
except ImportError:
    pass

import wikipedia
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import TokenTextSplitter
from neo4j import GraphDatabase
from pydantic import BaseModel, Field

try:
    from yfiles_jupyter_graphs import GraphWidget

    HAS_GRAPH_WIDGET = True
except ImportError:
    HAS_GRAPH_WIDGET = False

try:
    from IPython.display import display
except ImportError:
    display = print

wikipedia.set_user_agent("GraphRagPractice/1.0 (local learning project)")

C:\Users\23156\AppData\Local\Temp\ipykernel_9820\4228266941.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


C:\Users\23156\AppData\Local\Temp\ipykernel_9820\4228266941.py:18: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [14]:
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root (works when cwd is backend/ or project root)
_env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
for _env_path in _env_candidates:
    if _env_path.exists():
        load_dotenv(_env_path)
        print(f"Loaded env from {_env_path}")
        break
else:
    print("No .env file found — using system environment variables only")

# --- Ollama configuration ---
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
LLM_MODEL = os.environ.get("OLLAMA_LLM_MODEL", "llama3.2")
EMBED_MODEL = os.environ.get("OLLAMA_EMBED_MODEL", "nomic-embed-text")

# --- Neo4j configuration (from .env) ---
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

WIKIPEDIA_QUERY = os.environ.get("WIKIPEDIA_QUERY", "Imran Khan")

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
    base_url=OLLAMA_BASE_URL,
)
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

print(f"LLM: {LLM_MODEL} @ {OLLAMA_BASE_URL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Neo4j: {NEO4J_URI} (database: {NEO4J_DATABASE})")
print(f"Wikipedia topic: {WIKIPEDIA_QUERY}")

Loaded env from d:\projects\GraphRag\.env
LLM: llama3.2 @ http://localhost:11434
Embeddings: nomic-embed-text
Neo4j: neo4j+s://74ea21d8.databases.neo4j.io (database: neo4j)
Wikipedia topic: Imran Khan


In [15]:
from neo4j import GraphDatabase

# Quick connection test before building the LangChain graph wrapper
_test_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    connection_timeout=30,
)
try:
    _test_driver.verify_connectivity()
    with _test_driver.session(database=NEO4J_DATABASE) as _session:
        _session.run("RETURN 1").consume()
    print("Neo4j connection test passed")
except Exception as e:
    _test_driver.close()
    raise ConnectionError(
        "Could not connect to Neo4j Aura. "
        "If you see AuthError, reset your password at https://console.neo4j.io/ "
        "and update NEO4J_PASSWORD in .env. "
        f"Original error: {e}"
    ) from e
_test_driver.close()

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
print("Connected to Neo4j")

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct

## 1. Load and chunk Wikipedia documents

In [ ]:
import json
import requests

try:
    raw_documents = WikipediaLoader(query=WIKIPEDIA_QUERY, load_max_docs=3).load()
except (requests.exceptions.JSONDecodeError, json.JSONDecodeError) as e:
    print(f"Error loading Wikipedia page: {e}")
    raw_documents = []
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    raw_documents = []

print(f"Loaded {len(raw_documents)} documents")
for doc in raw_documents:
    print(f"  - {doc.metadata.get('title', '?')}")

In [ ]:
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])
print(f"Split into {len(documents)} chunks")

## 2. Extract knowledge graph with Ollama

In [ ]:
llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Extracted {len(graph_documents)} graph documents")
graph_documents

In [ ]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True,
)
print("Graph documents added to Neo4j")

## 3. Visualize the graph (optional)

In [ ]:
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"


def showGraph(cypher: str = default_cypher):
    if not HAS_GRAPH_WIDGET:
        print("yfiles_jupyter_graphs not available; skipping visualization.")
        return None

    driver = GraphDatabase.driver(
        uri=os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    )
    session = driver.session()
    widget = GraphWidget(graph=session.run(cypher).graph())
    widget.node_label_mapping = "id"
    display(widget)
    return widget


showGraph()

## 4. Build hybrid vector index with Ollama embeddings

In [ ]:
vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Hybrid vector index ready")

In [ ]:
graph.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]"
)

## 5. Entity extraction + structured retrieval

In [ ]:
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that appear in the text",
    )


entity_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following input: {question}",
        ),
    ]
)

entity_chain = entity_prompt | llm.with_structured_output(Entities)

In [ ]:
def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el["output"] for el in response])
    return result

In [ ]:
print(structured_retriever(f"Who is {WIKIPEDIA_QUERY}?"))

## 6. Hybrid retriever + RAG chain

In [ ]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [
        el.page_content for el in vector_index.similarity_search(question)
    ]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ".join(unstructured_data)}
    """
    return final_data

In [ ]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)


def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer


_search_query = RunnableBranch(
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | llm
        | StrOutputParser(),
    ),
    RunnableLambda(lambda x: x["question"]),
)

answer_template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

answer_prompt = ChatPromptTemplate.from_template(answer_template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | answer_prompt
    | llm
    | StrOutputParser()
)

## 7. Ask questions

In [ ]:
chain.invoke({"question": f"Who is {WIKIPEDIA_QUERY}?"})

In [ ]:
chain.invoke(
    {
        "question": "When did he become prime minister?",
        "chat_history": [
            (
                f"Who is {WIKIPEDIA_QUERY}?",
                "Imran Khan is a Pakistani former cricketer and politician who served as prime minister.",
            )
        ],
    }
)